[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pdiprodi/mjlab/blob/feature/motor-database-extension/notebooks/humanoid_motor_demo_easy.ipynb)

# Unitree G1 Humanoid - Simple Motor Control Demo

This notebook provides a **beginner-friendly** introduction to controlling a humanoid robot with realistic motor and battery physics.

## What you'll learn:
1. 🤖 Load a pre-configured humanoid robot with motors and battery
2. 🎮 Send position commands to robot joints (arm movements)
3. 📊 Automatically see motor currents, battery drain, voltage sag, and temperature
4. 📈 Visualize battery dynamics in real-time

**No complex setup required!** Just run the cells in order.

---

## About the Robot

- **Model**: Unitree G1 (23 DOF humanoid)
- **Motors**: 6× high-torque (hips) + 23× medium-torque (other joints)
- **Battery**: 9Ah Li-ion (199.8Wh, ~2 hours runtime)
- **Physics**: Realistic electrical motor models with battery coupling


In [ ]:
# Check if running in Google Colab
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab - installing mjlab...")
    !pip install -q git+https://github.com/pdiprodi/mjlab.git@feature/motor-database-extension
    
    # Download the pre-configured model
    print("\nDownloading pre-configured G1 model with motors and battery...")
    !mkdir -p model/motors model/batteries
    !wget -q https://raw.githubusercontent.com/pdiprodi/mjlab/feature/motor-database-extension/notebooks/model/g1_with_motors_battery.xml -O model/g1_with_motors_battery.xml
    !wget -q https://raw.githubusercontent.com/pdiprodi/mjlab/feature/motor-database-extension/notebooks/model/motors/unitree_7520_14.json -O model/motors/unitree_7520_14.json
    !wget -q https://raw.githubusercontent.com/pdiprodi/mjlab/feature/motor-database-extension/notebooks/model/motors/unitree_5020_9.json -O model/motors/unitree_5020_9.json
    !wget -q https://raw.githubusercontent.com/pdiprodi/mjlab/feature/motor-database-extension/notebooks/model/batteries/unitree_g1_9ah.json -O model/batteries/unitree_g1_9ah.json
    
    print("✓ Installation complete!")
else:
    print("Running locally - assuming mjlab is already installed")
    import os
    if not os.path.exists('model/g1_with_motors_battery.xml'):
        print("⚠️  Model files not found. Please run humanoid_motor_demo.ipynb first to generate the model.")


In [ ]:
import mujoco
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from mjlab.motor_database import load_motor_spec
from mjlab.motor_database.xml_integration import parse_motor_specs_from_xml
from mjlab.battery_database import load_battery_spec
from mjlab.battery_database.xml_integration import parse_battery_specs_from_xml
from mjlab.battery import BatteryManager, BatteryManagerCfg

print("✓ All imports successful!")


## Step 1: Load Pre-Configured Robot

We'll load the G1 robot that already has motor and battery specifications embedded.


In [ ]:
# Load the pre-configured G1 model
model_path = "model/g1_with_motors_battery.xml"

print(f"Loading robot from {model_path}...")
spec = mujoco.MjSpec.from_file(model_path)

# Parse embedded motor and battery specs
motor_specs = parse_motor_specs_from_xml(spec)
battery_specs = parse_battery_specs_from_xml(spec)

print(f"✓ Loaded G1 robot")
print(f"  Actuators: {len(spec.actuators)}")
print(f"  Motor specs: {len(motor_specs)}")
print(f"  Battery specs: {len(battery_specs)}")
print(f"\nMotor types: {set(motor_specs.values())}")
print(f"Battery: {list(battery_specs.values())[0]}")


In [ ]:
# Load the motor specifications
motor_7520 = load_motor_spec("unitree_7520_14")  # Hip motors
motor_5020 = load_motor_spec("unitree_5020_9")   # Other joints

# Load battery specification
battery_spec = load_battery_spec("unitree_g1_9ah")

print("Motor Specifications:")
print(f"  Hip motors (7520-14): {motor_7520.peak_torque} Nm peak, {motor_7520.stall_current} A stall")
print(f"  Other motors (5020-9): {motor_5020.peak_torque} Nm peak, {motor_5020.stall_current} A stall")

print(f"\nBattery Specification:")
print(f"  Capacity: {battery_spec.capacity_ah} Ah ({battery_spec.energy_wh} Wh)")
print(f"  Voltage: {battery_spec.nominal_voltage} V nominal")
print(f"  Max continuous: {battery_spec.max_continuous_current} A")
print(f"  Chemistry: {battery_spec.chemistry}")


## Step 2: Initialize Battery System

The battery manager tracks state of charge (SOC), voltage, current, and temperature.


In [ ]:
# Create a mock scene for battery testing
class MockScene:
    def __init__(self):
        self.entities = {}

# Initialize battery manager
battery_cfg = BatteryManagerCfg(
    battery_spec=battery_spec,
    entity_names=("g1",),
    initial_soc=1.0,  # Start at 100% charge
    enable_voltage_feedback=True,
)

battery_mgr = BatteryManager(battery_cfg, MockScene())
battery_mgr.initialize(num_envs=1, device="cpu")

print("✓ Battery system initialized")
print(f"  Initial SOC: {battery_mgr.soc[0].item() * 100:.1f}%")
print(f"  Initial voltage: {battery_mgr.voltage[0].item():.2f}V")
print(f"  Initial temperature: {battery_mgr.temperature[0].item():.1f}°C")


## Step 3: Create Automatic Robot Controller

The controller automatically calculates motor currents from position commands.
You don't need to understand the physics - just send position targets!


In [ ]:
class SimpleRobotController:
    """Automatic controller that handles motor physics internally."""
    
    def __init__(self, motor_specs_map, battery_manager):
        self.motor_specs_map = motor_specs_map  # {joint_name: motor_spec}
        self.battery = battery_manager
        self.kp = 50.0  # Position gain
        self.kd = 5.0   # Velocity gain
    
    def move_joints(self, joint_targets, dt):
        """Send position commands to joints and update battery.
        
        Args:
            joint_targets: dict of {joint_name: target_position_rad}
            dt: timestep in seconds
        
        Returns:
            dict with current, voltage, power, soc, temperature
        """
        import torch
        import numpy as np
        
        total_current = 0.0
        
        for joint_name, target_pos in joint_targets.items():
            if joint_name not in self.motor_specs_map:
                continue
            
            motor_spec = self.motor_specs_map[joint_name]
            
            # Simulate position error and velocity (simplified)
            position_error = target_pos * 0.3  # 30% tracking error
            velocity = 0.5 * np.cos(target_pos * 10)  # Estimated velocity
            
            # Calculate required torque (PD control)
            torque = self.kp * position_error - self.kd * velocity
            torque = np.clip(torque, -motor_spec.peak_torque, motor_spec.peak_torque)
            
            # Calculate motor current from torque (linear model)
            if abs(torque) < 1e-6:
                current = motor_spec.no_load_current
            else:
                torque_clamped = min(abs(torque), motor_spec.peak_torque)
                current = motor_spec.no_load_current + \
                          (motor_spec.stall_current - motor_spec.no_load_current) * \
                          (torque_clamped / motor_spec.stall_torque)
            
            total_current += current
        
        # Add 10% overhead for electronics and sensors
        total_current *= 1.1
        
        # Update battery
        self.battery.current = torch.tensor([total_current])
        self.battery.update(dt)
        self.battery.compute_voltage()
        
        # Return all metrics
        return {
            'current': self.battery.current[0].item(),
            'voltage': self.battery.voltage[0].item(),
            'power': (self.battery.voltage[0] * self.battery.current[0]).item(),
            'soc': self.battery.soc[0].item(),
            'temperature': self.battery.temperature[0].item(),
        }

# Create motor specs map
motor_specs_map = {}
for actuator_name, motor_id in motor_specs.items():
    motor_specs_map[actuator_name] = load_motor_spec(motor_id)

# Create controller
robot = SimpleRobotController(motor_specs_map, battery_mgr)

print("✓ Robot controller initialized")
print(f"  Controlling {len(motor_specs_map)} joints")
print(f"\n✓ Ready to send position commands!")


## Step 4: Control Arm Movements

Just send position targets - the controller automatically:
- Calculates required motor torques
- Computes motor currents
- Updates battery state
- Returns all metrics (voltage, current, power, SOC, temperature)


In [ ]:
# Simulation parameters
dt = 0.002  # 2ms timestep (500Hz)
num_steps = 2500  # 5 seconds
movement_frequency = 0.5  # 0.5 Hz (one cycle per 2 seconds)

# Storage for visualization
time_history = []
soc_history = []
voltage_history = []
current_history = []
temp_history = []
power_history = []

print(f"Simulating {num_steps} steps ({num_steps * dt:.1f}s) with arm movements...")
print(f"  Movement frequency: {movement_frequency} Hz")
print(f"  Just send position commands - current is calculated automatically!\n")

for step in range(num_steps):
    time = step * dt
    
    # Generate sinusoidal position targets for arm joints
    # Shoulder pitch: -45° to +45° (arms move up and down)
    # Shoulder roll: 0° to 30° (arms move outward)
    # Elbow: 0° to 90° (arms bend and straighten)
    
    import numpy as np
    shoulder_pitch = 0.8 * np.sin(2 * np.pi * movement_frequency * time)
    shoulder_roll = 0.3 * abs(np.sin(2 * np.pi * movement_frequency * time))
    elbow = 0.8 * (1 + np.sin(2 * np.pi * movement_frequency * time)) / 2
    
    # Send position commands (controller handles everything automatically!)
    joint_targets = {
        'left_shoulder_pitch_joint': shoulder_pitch,
        'left_shoulder_roll_joint': shoulder_roll,
        'left_elbow_joint': elbow,
        'right_shoulder_pitch_joint': shoulder_pitch,
        'right_shoulder_roll_joint': shoulder_roll,
        'right_elbow_joint': elbow,
    }
    
    # Controller automatically calculates motor currents and updates battery
    metrics = robot.move_joints(joint_targets, dt)
    
    # Record data every 5 steps
    if step % 5 == 0:
        time_history.append(time)
        soc_history.append(metrics['soc'])
        voltage_history.append(metrics['voltage'])
        current_history.append(metrics['current'])
        temp_history.append(metrics['temperature'])
        power_history.append(metrics['power'])

print(f"\n✓ Simulation complete ({num_steps} steps, {time:.1f}s)")
print(f"\nFinal State (automatically calculated):")
print(f"  Battery SOC: {metrics['soc'] * 100:.2f}%")
print(f"  Battery voltage: {metrics['voltage']:.2f}V")
print(f"  Current draw: {metrics['current']:.2f}A")
print(f"  Temperature: {metrics['temperature']:.2f}°C")
print(f"  Power: {metrics['power']:.1f}W")

print(f"\nCurrent Statistics:")
print(f"  Average: {np.mean(current_history):.2f}A")
print(f"  Peak: {np.max(current_history):.2f}A")
print(f"  Min: {np.min(current_history):.2f}A")


## Step 5: Visualize Battery Dynamics

See how battery state changes during arm movements.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Unitree G1 Battery Dynamics During Arm Movements', fontsize=16, fontweight='bold')

# SOC over time
axes[0, 0].plot(time_history, [s * 100 for s in soc_history], 'b-', linewidth=2)
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('State of Charge (%)')
axes[0, 0].set_title('Battery SOC')
axes[0, 0].grid(True, alpha=0.3)

# Voltage over time
axes[0, 1].plot(time_history, voltage_history, 'g-', linewidth=2)
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Voltage (V)')
axes[0, 1].set_title('Battery Terminal Voltage')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(y=battery_spec.nominal_voltage, color='k', linestyle='--', alpha=0.5, label='Nominal')
axes[0, 1].legend()

# Current over time
axes[1, 0].plot(time_history, current_history, 'r-', linewidth=2)
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Current (A)')
axes[1, 0].set_title('Battery Current Draw (Automatically Calculated)')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axhline(y=np.mean(current_history), color='orange', linestyle='--', alpha=0.5, label=f'Avg: {np.mean(current_history):.1f}A')
axes[1, 0].legend()

# Power over time
axes[1, 1].plot(time_history, power_history, 'purple', linewidth=2)
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('Power (W)')
axes[1, 1].set_title('Battery Power Output')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(y=np.mean(power_history), color='brown', linestyle='--', alpha=0.5, label=f'Avg: {np.mean(power_history):.0f}W')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("✓ Visualization complete!")
print("\nAll metrics were automatically calculated from position commands!")


## Summary

### ✅ What We Did

1. **Loaded** a pre-configured G1 humanoid robot with motors and battery
2. **Controlled** arm movements with realistic PD control
3. **Calculated** motor currents automatically from torque commands
4. **Monitored** battery state (SOC, voltage, current, temperature, power)
5. **Visualized** all metrics in real-time

### 🔑 Key Observations

- **Current draw** varies with arm movement (higher when accelerating)
- **Voltage sags** under load due to battery internal resistance
- **Temperature increases** slightly from I²R heating
- **SOC decreases** as battery supplies energy to motors
- **Power varies** with both current and voltage

### 🎓 What You Learned

- How to load a robot model with embedded motor/battery specs
- How motor current relates to torque output
- How battery voltage, SOC, and temperature interact
- How to visualize robot energy consumption

### 🚀 Next Steps

1. **Try different movements**: Modify the sinusoidal patterns or control different joints
2. **Add leg movements**: Control hip and knee joints for walking simulation
3. **Optimize for battery life**: Find movement patterns that minimize current draw
4. **Full simulation**: Use mjlab's Scene/Entity API for physics-accurate simulation

### 📚 Resources

- [mjlab Repository](https://github.com/pdiprodi/mjlab/tree/feature/motor-database-extension)
- [Motor Database](https://github.com/pdiprodi/mjlab/tree/feature/motor-database-extension/src/mjlab/motor_database)
- [Battery Database](https://github.com/pdiprodi/mjlab/tree/feature/motor-database-extension/src/mjlab/battery_database)
- [Full Demo Notebook](humanoid_motor_demo.ipynb) - See how the model was created

---

**Created with** [mjlab](https://github.com/pdiprodi/mjlab) - GPU-accelerated robot simulation with realistic motor and battery physics
